#Question 1 : Define Data Quality in the context of ETL pipelines. Why is it more than just data cleaning?
    -> Data Quality in ETL means ensuring that data is accurate, complete, consistent, valid, timely, and unique throughout the Extract, Transform, and Load process. It ensures the data is fit for business use and decision-making
    2.It is more than just data cleaning because:
    Data cleaning is only a small part of data quality. The difference is:
    Scope:
    Data cleaning = fixing errors like nulls, duplicates, typos after extraction.Data Quality = preventing errors at every stage of ETL using rules, validations, and monitoring.
    
    Timing:Cleaning is reactive. Data Quality is proactive and built into ETL design.
    
    Components:Data Quality includes:
    a. Accuracy – correct values from source.
    b. Completeness – no missing mandatory fields.
    c. Consistency – same format across systems.
    d. Validity – data follows business rules.
    e. Timeliness – data is loaded within SLA.
    f. Uniqueness – no duplicate records.
    
    Purpose:Cleaning improves data appearance. Data Quality ensures reliability of reports, analytics, and ML models.

#Question 2 : Explain why poor data quality leads to misleading dashboards and incorrect decisions.

    -> Poor data quality directly impacts dashboards and business decisions because dashboards are only as good as the data they show. This is called "Garbage In, Garbage Out".
    1.Misleading Dashboards:Dashboards visualize aggregated data. If the underlying data is wrong, the charts and KPIs will be wrong.  
    Inaccurate data: If sales amount = 0 or negative due to bad entry, the total revenue chart will show a false drop.  
    Incomplete data: If 30% of orders are missing because of extraction errors, the dashboard will under-report performance.  
    Inconsistent data: If Country = USA in one system and Country = US in another, grouping will split the same category, giving wrong totals.  
    Outdated data: If ETL job failed and last load was 5 days ago, the dashboard will show stale metrics.
    2.Incorrect Business Decisions: Executives and managers use dashboards to make decisions. Wrong data leads to wrong actions.  
    Financial loss: If inventory dashboard shows wrong stock due to duplicate records, company may over-order or under-stock.  
    Wrong strategy: If customer churn is under-reported due to missing records, marketing team may reduce retention campaigns.  Compliance risk: If financial dashboard uses invalid transaction amounts, it can cause audit failures or penalties.  
    Loss of trust: When users see incorrect numbers repeatedly, they stop using dashboards and go back to manual Excel reports.
      
#Question 3 : What is duplicate data? Explain three causes in ETL pipelines.
    -> 1. Definition of Duplicate Data:
    Duplicate data means having two or more identical or near-identical records for the same entity in a dataset. For example, the same customer appearing twice with CustomerID = 101. Duplicates waste storage, skew analytics, and cause inaccurate counts.2. Three Causes of Duplicate Data in ETL Pipelines:
    a. Multiple Source Systems:
    During extraction, data may come from different systems that store the same entity.
    Example: A customer registers on website and mobile app. Both systems send separate records with different IDs for the same person. ETL loads both without deduplication.
    b. ETL Job Re-run or Failure:
    If an ETL job fails mid-way and is re-run without proper checks, it can insert the same batch of records again.
    Example: A daily sales load inserts 10,000 rows. If the job is retried without "truncate" or "upsert" logic, the same 10,000 rows get loaded twice.
    c. Lack of Key Constraints / Poor Transformation Logic:
    During transformation, if primary keys are not defined or joins are not properly handled, duplicate rows are created.
    Example: Joining orders table with order_items table without grouping can create multiple rows for one order, making it look like multiple orders exist.

#Question 4 : Differentiate between exact, partial, and fuzzy duplicates.
    ->Exact Duplicates are records where all fields are completely identical. For example, if the same customer record with ID=101, Name=John, Age=25 appears twice in the dataset, it is an exact duplicate. These are the easiest to detect and remove using database constraints, DISTINCT, or GROUP BY operations.
    Partial Duplicates occur when some fields are identical but other fields are different or missing. For example, two records may have the same CustomerID and Name, but one has Age=25 and the other has Age=NULL. Partial duplicates often arise from incomplete data entry or multiple updates to the same record from different sources. They require matching on key fields to identify.
    Fuzzy Duplicates are records that refer to the same real-world entity but have small variations in spelling, formatting, or typos. For example, John Smith, 123 Main St and Jon Smyth, 123 Main Street are likely the same person but are not exact matches. Fuzzy duplicates are common when data comes from manual entry or different systems. Detecting them requires matching algorithms like Levenshtein Distance, Soundex, or similarity scoring.
    In ETL pipelines, handling all three types is important because even non-exact duplicates can cause over-counting, incorrect aggregation, and poor analytics if not resolved.

#Question 5 : Why should data validation be performed during transformation rather than after loading?
    ->Data validation should be performed during the Transformation phase of ETL rather than after loading because it prevents bad data from entering the target warehouse or database in the first place.
    1.Prevents Data Pollution:
    If validation is done after loading, invalid or dirty data is already stored in the warehouse. This pollutes the data lake or data warehouse and can affect all reports, dashboards, and downstream processes that consume it. Fixing it later requires rework and re-loading.
    2.Saves Time and Cost:
    Correcting errors after loading is expensive. You have to identify the bad records, roll back the load, fix the source or transformation logic, and re-run the entire ETL pipeline. Validation during transformation allows you to reject or quarantine bad records immediately, avoiding wasted compute and storage.
    3.Maintains Data Integrity and Business Rules:
    Transformation is where business rules, joins, calculations, and formatting are applied. This is the right stage to enforce rules like order_amount > 0, date must be in valid format, or customer_id must exist in master table. If validation is delayed, these rules cannot be enforced consistently.
    4.Improves Performance and Trust:
    Validating during transformation keeps the target warehouse clean and reliable. Users trust dashboards and analytics more when they know data has already been checked and cleansed before loading.

#Question 6 : Explain how business rules help in validating data accuracy. Give an example.
    -> Business rules are conditions or logic defined by the organization to ensure that data reflects real-world processes and constraints.
    In ETL, applying business rules during transformation is a key way to validate data accuracy, which means checking if the data correctly represents the intended meaning.
    How business rules help:  
    1.Enforce logical constraints: They check if values make sense in a business context, not just in format. This catches errors that basic type checks miss.  
    2.Detect anomalies: They flag records that violate expected patterns, so they can be rejected, corrected, or sent for review.
    3.Ensure consistency: They make sure data from different sources follows the same standards before loading.
    Example:
    Consider a sales ETL pipeline. A business rule might be: Order Quantity must be greater than 0 and Discount Percentage cannot be more than 50%.
    
    During transformation, if a record has Quantity = -5 or Discount = 70%, the ETL job will reject or flag it as invalid. Without this rule, the warehouse would store negative quantities or unrealistic discounts, making total revenue and profit calculations inaccurate.

#Question 7 :Write an SQL query on Sales_Transactions to list all duplicate keys and their counts using the business key (Customer_ID + Product_ID + Txn_Date + Txn_Amount )
    -> SELECT
    Customer_ID,
    Product_ID,
    Txn_Date,
    Txn_Amount,
    COUNT(*) AS Duplicate_Count
    FROM Sales_Transactions
    GROUP BY
    Customer_ID,
    Product_ID,
    Txn_Date,
    Txn_Amount
    HAVING COUNT(*) > 1;

#Question 8 : Enforcing Referential Integrity.Assume the following table:Identify Sales_Transactions.Customer_ID values that violate referential integrity when joined with Customers_Master and write a query to detect such violations.
    ->SELECT DISTINCT s.Customer_ID
    FROM Sales_Transactions s
    LEFT JOIN Customers_Master c
    ON s.Customer_ID = c.CustomerID
    WHERE c.CustomerID IS NULL;
